# Pelatihan Model & Evaluasi: LSTM
Di sini kita memuat data preprocessed dan melatih model LSTM raksasa.

In [21]:
import numpy as np
import pandas as pd
import time
import pickle
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Bidirectional, Dropout, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, CSVLogger
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load preprocessed data
data = np.load('processed_data.npz')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
TIME_STEPS = X_train.shape[1]
input_shape = (TIME_STEPS, X_train.shape[2])

with open('rps_scaler.pkl', 'rb') as f:
    rps_scaler = pickle.load(f)


## Arsitektur LSTM

In [22]:
model = Sequential([
    LSTM(128, activation='tanh', return_sequences=True, input_shape=input_shape),
    Dropout(0.2),
    LSTM(64, activation='tanh'),
    Dense(2)
], name="LSTM_Imdoukh")
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

/opt/jupyter-env/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "LSTM_Imdoukh"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_8 (LSTM)                   │ (None, 10, 128)        │        67,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,610 (455.51 KB)

 Trainable params: 116,610 (455.51 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
csv_logger = CSVLogger('training_progress_LSTM.csv', append=True)
EPOCHS = 200
BATCH_SIZE = 64

print(f"\n--- Training: LSTM ---")
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, csv_logger], verbose=1)



--- Training: LSTM ---
Epoch 1/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - loss: 0.0747 - mae: 0.2113 - val_loss: 0.0734 - val_mae: 0.2075
Epoch 2/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0725 - mae: 0.2082 - val_loss: 0.0706 - val_mae: 0.2023
Epoch 3/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0716 - mae: 0.2067 - val_loss: 0.0714 - val_mae: 0.2009
Epoch 4/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 0.0714 - mae: 0.2062 - val_loss: 0.0711 - val_mae: 0.2021
Epoch 5/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 0.0713 - mae: 0.2063 - val_loss: 0.0716 - val_mae: 0.1994
Epoch 6/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0715 - mae: 0.2065 - val_loss: 0.0712 - val_mae: 0.2048
Epoch 7/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0713 - mae: 0.2061 - val_loss: 0.0728 - val_mae: 0.2111
Epoch 8/200
199/199 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0712 - mae: 0.2061 - val_loss: 0.0723 - val_mae: 0.2069
Epoch 9/200
199/

## Evaluasi Kecepatan & Akurasi

In [24]:
single_sample = X_test[0:1]

# Pemanasan prediksi
_ = model.predict(single_sample, verbose=0)

# Precise inference benchmark
t_arr = []
for _ in range(30):
    t_s = time.time()
    model.predict(single_sample, verbose=0)
    t_arr.append(time.time() - t_s)
avg_inference_ms = np.mean(t_arr) * 1000

# Prediksi seluruh data test
pred = model.predict(X_test, verbose=0)
np.save('pred_LSTM.npy', pred)

results = {
    'LSTM': {
        'MSE': mean_squared_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'MAE': mean_absolute_error(y_test, pred),
        'R_Squared (R²)': r2_score(y_test, pred),
        'Speed (ms)': avg_inference_ms
    }
}
display(pd.DataFrame(results).T)


,MSE,RMSE,MAE,R_Squared (R²),Speed (ms)
LSTM,0.074134,0.272276,0.210808,0.424429,64.716538
